# Adaptive PID Control com RL — V2: Dual Network + MCTS

Extensão da V1 (DQN placeholder + PID fixo). Aqui o agente usa uma **rede dual (Policy + Value)** guiada por **Monte Carlo Tree Search (MCTS)** para planejar o ajuste dos ganhos PID a cada passo, em vez de uma Q-network vazia.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from env.agv_env import AGVEnv
from agent.network_v2 import DualPolicyValueNetwork
from agent.mcts_v2 import MCTS
from agent.pmcts_agent_v2 import PolicyValueMCTSAgent
from agent.gains_actions import apply_action
from evaluate import evaluate as evaluate_fixed_pid


## Arquitetura: Dual Network (Policy + Value)

Um torso MLP compartilhado extrai features do estado (erro, integral do erro, derivada do erro, tipo de trajetória) e alimenta duas cabeças: `policy_head` (prior de ação para o MCTS) e `value_head` (estimativa de retorno esperado, usada para avaliar folhas da árvore sem simular até o fim do episódio).

In [ ]:
network = DualPolicyValueNetwork()
print(network)


## MCTS guiado pela rede dual

A cada passo de decisão, o MCTS roda `n_simulations` simulações usando o próprio `AGVEnv` como modelo forward (via `copy.deepcopy`), guiado pelos priors da policy head e avaliando folhas com a value head (estilo PUCT/AlphaZero, adaptado para controle de único agente).

In [ ]:
env = AGVEnv()
agent = PolicyValueMCTSAgent(n_simulations=32)

state = env.reset()
gains = np.array([0.1, 0.01, 0.001])
action, action_probs = agent.select_action(env, state, gains)
print('Ação escolhida:', action)
print('Distribuição de visitas do MCTS:', np.round(action_probs, 3))


## Loop de Treinamento V2

Treino por "auto-jogo": cada episódio gera (estado, distribuição do MCTS, retorno descontado), usados para treinar a rede dual (policy via cross-entropy contra a distribuição do MCTS, value via MSE contra o retorno observado).

In [ ]:
from train_v2 import run_episode, discount_returns

episodes = 50
history_v2 = []

for ep in range(episodes):
    ep_states, ep_probs, ep_rewards = run_episode(env, agent)
    returns = discount_returns(ep_rewards)
    stats = agent.train_step(ep_states, ep_probs, returns)
    history_v2.append({'episode': ep, 'return': float(np.sum(ep_rewards)), **stats})
    if ep % 10 == 0:
        print(f"Ep {ep:03d} | Return: {history_v2[-1]['return']:.3f} | Loss: {stats['loss']:.4f}")

agent.save('agent_v2.pt')


## Resultados Iniciais

In [ ]:
returns_v2 = [h['return'] for h in history_v2]
plt.plot(returns_v2)
plt.xlabel('Episódio')
plt.ylabel('Retorno acumulado')
plt.title('V2 - Retorno por episódio (MCTS + Rede Dual)')
plt.show()


## Benchmark V1 vs V2

In [ ]:
fixed_pid = np.array([0.1, 0.01, 0.001])
mean_fixed, std_fixed = evaluate_fixed_pid(fixed_pid)

from evaluate_v2 import evaluate_v2
mean_v2, std_v2 = evaluate_v2(agent)

print(f'V1 - PID Fixo         | Mean Error: {mean_fixed:.4f} | Std: {std_fixed:.4f}')
print(f'V2 - MCTS + Rede Dual | Mean Error: {mean_v2:.4f}   | Std: {std_v2:.4f}')


## Gráfico Comparativo V1 vs V2

In [ ]:
methods = ['PID Fixo (V1)', 'DQN placeholder (V1)', 'MCTS + Rede Dual (V2)']
errors = [mean_fixed, mean_fixed, mean_v2]  # DQN da V1 não tinha policy real; reaproveita PID fixo como referência

plt.figure(figsize=(7, 5))
plt.bar(methods, errors)
plt.ylabel('Erro Médio Absoluto')
plt.title('Comparação V1 x V2')
plt.xticks(rotation=15)
plt.show()


## Convergência do Aprendizado (Erro Suavizado)

In [ ]:
import pandas as pd

losses = pd.Series([h['loss'] for h in history_v2])
rolling_loss = losses.rolling(window=10).mean()

plt.figure(figsize=(8, 5))
plt.plot(rolling_loss)
plt.xlabel('Episódio')
plt.ylabel('Loss (média móvel)')
plt.title('V2 - Convergência do treino (policy + value loss)')
plt.show()


## Conclusão

A V2 troca a Q-network placeholder da V1 por uma rede dual treinada com alvos gerados por MCTS, permitindo planejamento explícito sobre o modelo do ambiente antes de cada ajuste de ganho — a mesma ideia central de AlphaZero, aplicada a um problema de controle de único agente.